<a href="https://colab.research.google.com/github/Rajarajan1087/fde-learning-lab/blob/FinanceFlow/FDE%20Projects/FinanceFlow/PyNotes/financeflow_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# Generate Pandas and load google drive
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Cell 1 :Build the Default Risk Classifier

9.	Load financeflow_clean.csv. One-hot encode sector, loan_purpose, state.
10.	Features: business_age_years, num_employees, annual_revenue_usd, loan_amount_usd, loan_term_months, interest_rate_pct, credit_score, debt_to_income_ratio, plus encoded categoricals.
11.	Split 80/20 train/test, random_state=42.
12.	Train a Random Forest Classifier (n_estimators=200, random_state=42, class_weight='balanced').
13.	Print: accuracy, precision, recall, F1, ROC-AUC. In a markdown cell, explain why class_weight='balanced' matters for a dataset with a 14% default rate.
14.	Score all 330 loans. Add columns: default_probability (float) and decision (Decline if >= 0.65, Review if 0.40–0.64, Approve if < 0.40).


In [17]:


df = pd.read_csv('/content/drive/MyDrive/FDE Projects/FinanceFlow/Data/financeflow_clean.csv')
print(df.shape)
# print(df.head())
print(df.columns.tolist())
print(df.dtypes)
print(df.isnull().sum())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(330, 13)
['loan_id', 'business_age_years', 'sector', 'num_employees', 'annual_revenue_usd', 'loan_amount_usd', 'loan_term_months', 'interest_rate_pct', 'credit_score', 'debt_to_income_ratio', 'loan_purpose', 'state', 'default_flag']
loan_id                  object
business_age_years        int64
sector                   object
num_employees             int64
annual_revenue_usd      float64
loan_amount_usd         float64
loan_term_months          int64
interest_rate_pct       float64
credit_score              int64
debt_to_income_ratio    float64
loan_purpose             object
state                    object
default_flag              int64
dtype: object
loan_id                 0
business_age_years      0
sector                  0
num_employees           0
annual_revenue_usd      0
loan_amount_usd         0
loan_term_months        0
interest_rate_pct       0

In [18]:
# Checking  class balance — how many defaults vs non-defaults.
print(df['default_flag'].value_counts())
print(df['default_flag'].value_counts(normalize=True).round(3))

default_flag
0    238
1     92
Name: count, dtype: int64
default_flag
0    0.721
1    0.279
Name: proportion, dtype: float64


# Cell 2 -  Default rate increases after incorrect data removal ❌

10.	Features: business_age_years, num_employees, annual_revenue_usd, loan_amount_usd, loan_term_months, interest_rate_pct, credit_score, debt_to_income_ratio, plus encoded categoricals.

Note :

 - 238 non-defaults vs 92 defaults
 - Roughly 2.6:1 imbalance
 - Adding up one-hot encode the categorical columns.





In [19]:
df_encoded = pd.get_dummies(df, columns=['sector', 'loan_purpose', 'state'], drop_first=False)
df_encoded = df_encoded.drop(columns=['loan_id'])
print(df_encoded.shape)

(330, 31)


In [20]:
X = df_encoded.drop(columns=['default_flag'])
y = df_encoded['default_flag']

print(X.shape)
print(y.shape)

(330, 30)
(330,)


# CELL 3 : Training data Set up ⚛

  - 11.	Split 80/20 train/test, random_state=42.

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape)
print(X_test.shape)

(264, 30)
(66, 30)


# Cell 4 : Train Model ⚖

  -	Train a Random Forest Classifier (n_estimators=200, random_state=42, class_weight='balanced').

In [22]:
from sklearn.ensemble import RandomForestClassifier
import joblib
import os
from google.colab import drive

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train, y_train)
print("Model trained successfully")

joblib.dump(model, '/content/drive/MyDrive/FDE Projects/FinanceFlow/TrainedModels/financeflow_model.pkl')
print("Model saved successfully")


Model trained successfully
Model saved successfully


# Cell 5 : Model evaluation ✅

  - Print: accuracy, precision, recall, F1, ROC-AUC.
  - class_weight='balanced' tells the model — "don't ignore defaults just because they're fewer in number. Treat each default as 2.6x more important than a non-default."
  - Score all 330 loans. Add columns: default_probability (float) and decision (Decline if >= 0.65, Review if 0.40–0.64, Approve if < 0.40).

In [23]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall   :", round(recall_score(y_test, y_pred), 3))
print("F1 Score :", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob), 3))

Accuracy : 0.848
Precision: 1.0
Recall   : 0.412
F1 Score : 0.583
ROC-AUC  : 0.826


In [24]:
df_encoded['default_probability'] = model.predict_proba(df_encoded.drop(columns=['default_flag']))[:, 1]

df_encoded['decision'] = df_encoded['default_probability'].apply(
    lambda x: 'Decline' if x >= 0.65 else ('Review' if x >= 0.40 else 'Approve')
)

print(df_encoded['decision'].value_counts())
print(df_encoded[['default_probability', 'decision']].head(10))

decision
Approve    240
Decline     74
Review      16
Name: count, dtype: int64
   default_probability decision
0                0.060  Approve
1                0.135  Approve
2                0.085  Approve
3                0.260  Approve
4                0.075  Approve
5                0.700  Decline
6                0.715  Decline
7                0.105  Approve
8                0.070  Approve
9                0.275  Approve


In [25]:
df_scored = df.copy()
df_scored['default_probability'] = df_encoded['default_probability'].values
df_scored['decision'] = df_encoded['decision'].values

df_scored.to_csv('/content/drive/MyDrive/FDE Projects/FinanceFlow/Data/financeflow_scored.csv', index=False)
print(df_scored.shape)
print(df_scored[['loan_id', 'default_probability', 'decision']].head())

(330, 15)
  loan_id  default_probability decision
0  L00001                0.060  Approve
1  L00002                0.135  Approve
2  L00003                0.085  Approve
3  L00004                0.260  Approve
4  L00005                0.075  Approve


In [26]:
# "Print the top 10 features driving default risk."

import pandas as pd

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(10))

                     feature  importance
0         business_age_years    0.142600
6               credit_score    0.130368
7       debt_to_income_ratio    0.118598
3            loan_amount_usd    0.089604
2         annual_revenue_usd    0.089059
1              num_employees    0.086035
5          interest_rate_pct    0.083705
4           loan_term_months    0.042510
27                  state_PA    0.013598
17  loan_purpose_Real Estate    0.012932


### Top 10 Features Driving Default Risk

The three strongest predictors of default are:

1. **Business Age** (14.3%) — younger businesses default more frequently
2. **Credit Score** (13.0%) — lower scores correlate strongly with default
3. **Debt-to-Income Ratio** (11.9%) — higher debt burden relative to income increases risk

Loan amount and annual revenue follow, suggesting loan sizing relative
to business capacity matters. Notably, geographic (state_PA) and
loan purpose (Real Estate) signals appear in the top 10 but contribute
minimally — the model is primarily driven by financial health indicators,
not categorical attributes.

In [28]:
# TO verify created model and scored columns details are intact to work on agent details
print(df_scored.shape)
print(model)
print(df_scored['decision'].value_counts())

(330, 15)
RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)
decision
Approve    240
Decline     74
Review      16
Name: count, dtype: int64


# Cell 6 :  LangChain Automated Underwriter Brief -  API Set up

For every loan marked Decline or Review, use LangChain and the **(Groq APi been used in this assgmnet due to billing issues)** Claude API to generate an automated underwriter brief — a short, structured note that tells the analyst exactly why the model flagged the application.


19.	Export Decline and Review records to financeflow_decisions_today.csv with columns: loan_id, sector, loan_amount_usd, credit_score, debt_to_income_ratio, default_probability, decision, underwriter_brief.


In [5]:
# Anthropic api access installation
# !pip install anthropic -q

# from google.colab import userdata
# import anthropic

# client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
# response = client.messages.create(
#     model="claude-sonnet-4-6", max_tokens=100,
#     messages=[
#         {"role": "user", "content": "Say 'FinanceFlow API connection successful' and nothing else."}
#     ]
# )
# print(response.content[0].text)

# Anthropic API testng mock response due to paymenty issus
def mock_claude_response(loan_id, credit_score, dti, decision):
    return f"Risk Brief: Loan {loan_id} has been flagged as {decision}. Credit score of {credit_score} and debt-to-income ratio of {dti} were primary risk indicators."

In [13]:
# Mocking api with groq instead of claude
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.5 MB/s eta 0:00:00


In [14]:
from groq import Groq
from google.colab import userdata

client_llm = Groq(api_key=userdata.get('GROQ_API_KEY'))

response = client_llm.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say 'FinanceFlow API connection successful' and nothing else."}
    ]
)
print(response.choices[0].message.content)

FinanceFlow API connection successful


# Cell 7 : LangChain Automated Underwriter Brief - Agent Set up

16.	PromptTemplate variables: loan_id, business_age_years, sector, loan_amount_usd, credit_score, debt_to_income_ratio, default_probability, decision.
17.	The prompt must produce a 3-part structured brief:

    •	**Risk Summary:** one sentence stating the decision and top risk factor.
    •	**Key Signals:** two bullet points — the two data points most driving the risk score.
    •	**Analyst Action:** one sentence recommending the next step (e.g. request 3 months bank statements, verify revenue with accountant letter).

18.	Do not use financial jargon the business owner would not understand. The brief must be readable by the applicant if shared.

In [19]:
!pip install langchain-core -q

In [24]:
# Loading DF_Scored for prompt testing
df_scored = pd.read_csv('/content/drive/MyDrive/FDE Projects/FinanceFlow/Data/financeflow_scored.csv')
print(df_scored.shape)

(330, 15)


In [27]:
from langchain_core.prompts import PromptTemplate

brief_template = PromptTemplate(
    input_variables=[
        "loan_id", "business_age_years", "sector", "loan_amount_usd",
        "credit_score", "debt_to_income_ratio", "default_probability", "decision"
    ],
    template="""
You are an underwriter assistant for FinanceFlow Capital.

Based on the loan application data below, write a short underwriter brief.
Use plain English — no financial jargon. The brief must be readable by the business owner.

Loan Data:
- Loan ID: {loan_id}
- Business Age: {business_age_years} years
- Sector: {sector}
- Loan Amount: ${loan_amount_usd}
- Credit Score: {credit_score}
- Debt-to-Income Ratio: {debt_to_income_ratio}
- Default Probability: {default_probability}
- Decision: {decision}

Write the brief in exactly this structure:

Risk Summary: [One sentence stating the decision and the top risk factor]

Key Signals:
- [First data point driving the risk]
- [Second data point driving the risk]

Analyst Action: [One concrete document or verification step the analyst should request — for example: request 3 months bank statements, verify revenue with an accountant letter, or confirm outstanding liabilities with a credit bureau check]
"""
)

print("PromptTemplate ready")

PromptTemplate ready


In [28]:
# Prompt Testing - Single declined loan data

def generate_brief(row):
    prompt = brief_template.format(
        loan_id=row['loan_id'],
        business_age_years=row['business_age_years'],
        sector=row['sector'],
        loan_amount_usd=row['loan_amount_usd'],
        credit_score=row['credit_score'],
        debt_to_income_ratio=row['debt_to_income_ratio'],
        default_probability=round(row['default_probability'], 3),
        decision=row['decision']
    )

    response = client_llm.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

# Test on one loan first
test_row = df_scored[df_scored['decision'] == 'Decline'].iloc[0]
brief = generate_brief(test_row)
print(brief)

Risk Summary: We are declining the loan application for Loan ID L00006 due to the high default probability of 0.7, which indicates a significant risk that the business may not be able to repay the loan.

Key Signals:
- The business has a relatively high debt-to-income ratio of 0.527, which suggests that a large portion of its income is already going towards debt repayment, leaving limited room for additional loan payments.
- The credit score of 580 is also a concern, as it indicates a history of credit issues and increases the likelihood of default.

Analyst Action: Verify outstanding liabilities with a credit bureau check to confirm the business's current debt obligations and assess its ability to manage additional debt.


In [29]:
# Prompt Testing - 90 flagged(decline/ review) loan data
flagged_loans = df_scored[df_scored['decision'].isin(['Decline', 'Review'])].copy()
print(f"Generating briefs for {len(flagged_loans)} loans...")

flagged_loans['underwriter_brief'] = flagged_loans.apply(generate_brief, axis=1)
print("Done")

Generating briefs for 90 loans...
Done


In [30]:
# Exporting into CSV
output_cols = [
    'loan_id', 'sector', 'loan_amount_usd', 'credit_score',
    'debt_to_income_ratio', 'default_probability', 'decision', 'underwriter_brief'
]

financeflow_decisions = flagged_loans[output_cols].copy()
financeflow_decisions.to_csv('/content/drive/MyDrive/FDE Projects/FinanceFlow/Data/financeflow_decisions_today.csv', index=False)

print(f"Exported {len(financeflow_decisions)} records")
print(financeflow_decisions[['loan_id', 'decision', 'underwriter_brief']].head(3))

Exported 90 records
   loan_id decision                                  underwriter_brief
5   L00006  Decline  Risk Summary: We are declining loan applicatio...
6   L00007  Decline  Risk Summary: We have decided to decline loan ...
14  L00045  Decline  Risk Summary: We have decided to decline the l...
